# Task 3: Cleaning Data (Dirty Cafe Sales)

**Goal:** Turn a deliberately messy cafe sales dataset (10,000 rows, 8 columns) into a clean, analysis-ready one, and explain every decision.

**Dataset:** Dirty Cafe Sales (Kaggle). Columns: Transaction ID, Item, Quantity, Price Per Unit, Total Spent, Payment Method, Location, Transaction Date.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("dirty_cafe_sales.csv")
df_raw = df.copy()      # kept for the before/after table
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [2]:
print("Shape:", df.shape)
print()
df.info()

Shape: (10000, 8)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [3]:
print("Nulls per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

Nulls per column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicate rows: 0


In [4]:
for col in df.select_dtypes("object").columns:
    if df[col].nunique() < 30:
        print(f"--- {col} ---")
        print(df[col].value_counts(dropna=False))
        print()

--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

--- Quantity ---
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

--- Price Per Unit ---
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164
Name: count, dtype: int64

--- Total Spent ---
Total Spent
6.0        979
12.0       939
3.0        930
4.0        923
20.0       746
15.0       734
8.0        677
10.0       524
2.0        497
9.0        479
5.0        468
16.0       444
25.0       259
7.5        237
1.0        232
4.5        225
1.5        205
NaN        173
UNKNOWN    165
ERROR      164
Name: count, dtype: int64

--- Payment Method 

In [5]:
df.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


## Data Quality Report

- **Shape:** 10,000 rows × 8 columns.
- **Data types:** all 8 columns are stored as text (`object`), including the numbers and dates. The text placeholders "ERROR" and "UNKNOWN" caused this.
- **Duplicates:** 0 duplicate rows, and all 10,000 Transaction IDs are unique.
- **Hidden missing values:** `isnull()` only finds real nulls. Many cells contain the words "ERROR" or "UNKNOWN", which look like data but are not.

| Column | Real nulls | "ERROR" | "UNKNOWN" | Total unusable | % of rows |
|---|---|---|---|---|---|
| Item | 333 | 292 | 344 | 969 | 9.7% |
| Quantity | 138 | 170 | 171 | 479 | 4.8% |
| Price Per Unit | 179 | 190 | 164 | 533 | 5.3% |
| Total Spent | 173 | 164 | 165 | 502 | 5.0% |
| Payment Method | 2,579 | 306 | 293 | 3,178 | 31.8% |
| Location | 3,265 | 358 | 338 | 3,961 | 39.6% |
| Transaction Date | 159 | (mixed in) | (mixed in) | 460 | 4.6% |

In total there are **10,082 missing or invalid cells**. Payment Method and Location are the worst, with roughly a third or more of their values unusable.

In [6]:
df = df_raw.copy()
df = df.replace(["ERROR", "UNKNOWN"], np.nan)      # treat placeholders as real nulls
for c in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[c] = pd.to_numeric(df[c])

print(df.groupby("Item")["Price Per Unit"].agg(["nunique", "min", "max"]))

          nunique  min  max
Item                       
Cake            1  3.0  3.0
Coffee          1  2.0  2.0
Cookie          1  1.0  1.0
Juice           1  3.0  3.0
Salad           1  5.0  5.0
Sandwich        1  4.0  4.0
Smoothie        1  4.0  4.0
Tea             1  1.5  1.5


## Cleaning Approach

**Step 1: Make the hidden missing values visible.** I replaced "ERROR" and "UNKNOWN" with real missing values (NaN) so they can be handled like any other null, then converted Quantity, Price Per Unit and Total Spent to numbers.

**Step 2: Check the menu.** Each item has exactly one price (Cookie 1.0, Tea 1.5, Coffee 2.0, Cake 3.0, Juice 3.0, Sandwich 4.0, Smoothie 4.0, Salad 5.0). Also, Total Spent = Quantity × Price Per Unit. This means many missing numbers can be recovered exactly instead of guessed.

In [7]:
price_map = df.dropna(subset=["Item", "Price Per Unit"]).groupby("Item")["Price Per Unit"].agg(lambda s: s.mode()[0])

df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Total Spent"] / df["Quantity"])
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(price_map))
df["Quantity"]       = df["Quantity"].fillna(df["Total Spent"] / df["Price Per Unit"])
df["Total Spent"]    = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])

print(df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().sum())

Quantity          23
Price Per Unit     6
Total Spent       23
dtype: int64


In [8]:
known = df.dropna(subset=["Item", "Price Per Unit"])
items_per_price = known.groupby("Price Per Unit")["Item"].nunique()
single_prices = items_per_price[items_per_price == 1].index
price_to_item = known[known["Price Per Unit"].isin(single_prices)].groupby("Price Per Unit")["Item"].first()
print(price_to_item)

mask = df["Item"].isna()
df.loc[mask, "Item"] = df.loc[mask, "Price Per Unit"].map(price_to_item)
print("Items still missing:", df["Item"].isna().sum())

Price Per Unit
1.0    Cookie
1.5       Tea
2.0    Coffee
5.0     Salad
Name: Item, dtype: object
Items still missing: 480


In [9]:
df[["Item", "Payment Method", "Location"]] = df[["Item", "Payment Method", "Location"]].fillna("Unknown")

before = len(df)
df = df.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])
print("Rows dropped:", before - len(df))

Rows dropped: 26


## Missing Data Strategy (one decision per column)

| Column | Strategy | Why |
|---|---|---|
| Quantity, Price Per Unit, Total Spent | Recomputed using Total = Quantity × Price, and the item's fixed menu price | Gives the true value with no guessing. Imputing a mean would have created values that don't match the formula. |
| Item | Filled from the price only when that price belongs to one item (1.0 Cookie, 1.5 Tea, 2.0 Coffee, 5.0 Salad). Otherwise set to "Unknown". | Prices 3.0 (Cake/Juice) and 4.0 (Sandwich/Smoothie) are shared, so the item can't be identified. About 480 rows stay "Unknown". |
| Payment Method | Filled with "Unknown" | About 32% was missing. Filling with the most common value (mode) would invent a fake majority and distort the payment analysis. |
| Location | Filled with "Unknown" | About 40% was missing, so the same reasoning applies. |
| Transaction Date | Left as missing (NaT) | A made-up date would damage any time-based analysis. 460 dates remain missing. |
| Rows where numbers could not be recovered | Dropped | 26 rows (0.26% of the data) had too many numeric fields missing to recompute. This is a very small loss. |

In [10]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
print("Missing dates:", df["Transaction Date"].isna().sum())

print("Duplicate rows:", df.duplicated().sum(), "| duplicate IDs:", df["Transaction ID"].duplicated().sum())

# validation: does Quantity x Price = Total on every row?
print("Rows where Qty x Price != Total:", (df["Quantity"] * df["Price Per Unit"] != df["Total Spent"]).sum())

Missing dates: 460
Duplicate rows: 0 | duplicate IDs: 0
Rows where Qty x Price != Total: 0


## Duplicates and Validation

- **Duplicates:** I checked for duplicate rows and for duplicate Transaction IDs. **None were found (0 and 0)**, so nothing was removed.
- **Standardisation:** the inconsistent values in this dataset are the placeholders "ERROR" and "UNKNOWN" (the equivalent of mixed spellings like "Male"/"M"). They were converted to proper missing values, and the dates were converted to `datetime`.
- **Validation:** after cleaning, I checked every row. **0 rows** have Quantity × Price Per Unit different from Total Spent, so the numeric columns are consistent.

In [11]:
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: bounds [{low}, {high}], outliers = {n}")

Quantity: bounds [-1.0, 7.0], outliers = 0
Price Per Unit: bounds [-1.0, 7.0], outliers = 0
Total Spent: bounds [-8.0, 24.0], outliers = 268


## Outlier Detection

I used the IQR method (values outside Q1 - 1.5×IQR and Q3 + 1.5×IQR) on Quantity, Price Per Unit and Total Spent.

| Column | Bounds | Outliers |
|---|---|---|
| Quantity | [-1, 7] | 0 |
| Price Per Unit | [-1, 7] | 0 |
| Total Spent | [-8, 24] | 268 |

All 268 flagged values in Total Spent are exactly 25.0, which is the maximum possible purchase (5 units at a price of 5.0). Since Total Spent equals Quantity × Price Per Unit and every row passed that check, these are real sales, not data errors. The IQR method flagged them only because large totals are rare. **Decision: retain all of them.** Capping or removing them would delete valid transactions and understate the cafe's biggest sales.

In [12]:
df["Transaction ID"] = df["Transaction ID"].astype("string")
df["Quantity"] = df["Quantity"].astype(int)
for c in ["Item", "Payment Method", "Location"]:
    df[c] = df[c].astype("category")
df.dtypes

Transaction ID      string[python]
Item                      category
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[ns]
dtype: object

## Data Type Correction

| Column | Before | After |
|---|---|---|
| Transaction ID | object | string |
| Item, Payment Method, Location | object | category |
| Quantity | object | int64 |
| Price Per Unit, Total Spent | object | float64 |
| Transaction Date | object | datetime64 |

Every column now has an appropriate type. Quantity is a whole number, money values are floats, and the repeated text labels are categories.

In [13]:
def bad_count(frame, col):
    return frame[col].isna().sum() + frame[col].isin(["ERROR", "UNKNOWN"]).sum()

summary = pd.DataFrame({
    "Before": [len(df_raw), df_raw.duplicated().sum(), sum(bad_count(df_raw, c) for c in df_raw.columns), (df_raw.dtypes == "object").sum()],
    "After":  [len(df), df.duplicated().sum(), df.isna().sum().sum(), (df.dtypes == "object").sum()],
}, index=["Rows", "Duplicate rows", "Missing/invalid cells", "Columns stored as text (object)"])
summary

,Before,After
Rows,10000,9974
Duplicate rows,0,0
Missing/invalid cells,10082,460
Columns stored as text (object),8,0


## Before vs After Cleaning

| Metric | Before | After |
|---|---|---|
| Rows | 10,000 | 9,974 |
| Duplicate rows | 0 | 0 |
| Missing/invalid cells | 10,082 | 460 |
| Columns stored as text (object) | 8 | 0 |

**How to read this table:** missing or invalid cells fell by about 95%. The remaining 460 are all in Transaction Date, which I left missing on purpose. The "Unknown" labels in Item, Payment Method and Location are intentional categories, so they are not counted as missing here.

In [14]:
print(df.loc[(df["Total Spent"] < -8) | (df["Total Spent"] > 24), "Total Spent"].value_counts())

Total Spent
25.0    268
Name: count, dtype: int64


## Conclusion

The raw dataset had no duplicates but had 10,082 missing or invalid cells, and every column was stored as text. After cleaning, 9,974 of 10,000 rows remain (99.74%), all numeric columns are correct and consistent, and the data types are right. The cleaned file is saved as `cleaned_cafe_sales.csv`.

The main idea was to **recover values using the data's own rules** (Total = Quantity × Price, and fixed menu prices) instead of deleting rows or filling averages.

**Limitations:**
- About 480 rows have an unknown Item, because some menu items share a price.
- 460 transactions have no usable date, so any time-based analysis must exclude them.
- Payment Method and Location are "Unknown" for a large share of rows, so results using them should be treated with caution.